In [ ]:
import os
import sys

sys.path.append("../")

envkey = "OMP_NUM_THREADS"
# Set this environment variable to the number of available cores in your machine,
# to get a fast execution of the Einstein Boltzmann Solver
print("The value of {:s} is: ".format(envkey), os.environ.get(envkey))
os.environ[envkey] = str(12)
os.environ[envkey] = str(12)
print("The value of {:s} is: ".format(envkey), os.environ.get(envkey))

In [ ]:
import astropy.units as u
import matplotlib.pyplot as plt
import numpy as np
import seaborn
import niceplots.utils as nicepl

import numpy as np
import matplotlib.pyplot as plt
from scipy.special import roots_legendre, spherical_jn, j1
from matplotlib.colors import LinearSegmentedColormap
import matplotlib as mpl

nicepl.initPlot()
Cs = seaborn.color_palette("colorblind")
Cp = seaborn.color_palette("Paired")
Cs

In [ ]:
from scipy.optimize import curve_fit

def linear(x, a, b):
    return a * x + b

In [ ]:
# Import Main modules. This might take some time as some functions compile before time
from SSLimPy.interface import sslimpy
from SSLimPy.interface import survey_specs
from SSLimPy.LIMsurvey import covariance as scov
from SSLimPy.LIMsurvey import power_spectrum as spobs

In [ ]:
settings = {
    "code":"class", # The Einstein--Boltzman solver that should be used
    "do_RSD" : False, # If RSD should be considerd
    "nonlinearRSD" : False, # If you want to add FOG to the RSD
    "QNLpowerspectrum": False, # Use dewiggled power spectrum (vlasov approximation of nonlinear structure formation)
    "FoG_damp" : "ISTF_like", # The particular parametrization for the FOG. Check PowerSpectrum for the full list
    "halo_model_PS" : True, # If the cosmological shotnoise should be computed from the halo model 
    "output" : ["Power spectrum", "Covariance"], # What output one wants (here power spectrum and Gaussian covariance only)
    "kmin": 1e-4 * u.Mpc**-1,
    "kmax": 50 * u.Mpc**-1,
    "nk": 200,
    "Smooth_resolution": True,
    "nonlinearMatpow": False,
}

h = 0.6736
Omegab = 0.02237 / h**2
Omegam = 0.3153
ns = 0.9649
log109As = 3.044
mnu = 0.06

cosmodict={
    "h":h,
    "Omegab":Omegab,
    "Omegam":Omegam,
    "mnu":mnu,
    "ln_A_s_1e10": log109As,
    "ns":ns,
}

# Parameters that enter your halo model. Typically they are not changed but you could
halodict={
    "halo_tracer" : "clustering", # Computes all halo quantities from the matter field - neutrinos
    "hmf_model": "ST", # Sheth--Tormann halo mass function
    "concentration": "Diemer19", # Diemer19 halo concentration relation
    "bias_model": "ST99",
    "nonlinear_bias": "HMF",
}

In [ ]:
myssl = sslimpy.SSLimPy(
    settings_dict=settings,
    cosmopars=cosmodict,
    halopars=halodict,
)

In [ ]:
mycosmo = myssl.current_cosmology
myhalo = myssl.current_halomodel

In [ ]:
nu = 1.901 * u.THz

nuObs = 300 * u.GHz
FWHMnu = nuObs / 100
dnu = FWHMnu / np.sqrt(8 * np.log(2))

normalisation = {
    "Omega_field" : 1 * u.deg**2,
    "Delta_nu" : 10 * u.GHz,
    "nuObs" : nuObs,
    "dnu" : dnu,
}

In [ ]:
z = (nu / nuObs -1).to(1).value
z = np.atleast_1d(z)
z

In [ ]:
astrodict_CCATp={
    "model_type": "ML",
    "model_name": "SilvaCII",
    "model_par": {
        "a": 0.8475,
        "b": 7.2203,
        "SFR_file": "sfr_release.dat",
        "do_quench": False,
    },
    "sigma_scatter" : 0.37,
}

surveyspecs_CCATp = {
    "Tsys_NEFD": 0 * u.uK, #81 * u.mJy * u.s**0.5, #/ np.sqrt(6912)
    "Nfeeds": 6912,
    "nD": 1,
    "beam_FWHM": 48 * u.arcsec,
    "nu": nu,
    "tobs": 200000 / 42 * u.h,
    # "do_Jysr": True,
}

In [ ]:
surveyspecs_CCATp.update(normalisation)
surveyspecs_CCATp

In [ ]:
specs = surveyspecs_CCATp.copy()
pobs_CII = myssl.compute(
    myssl.current_cosmology.cosmopars,
    myssl.current_halomodel.haloparams,
    astrodict_CCATp,
    specs,
    pobs_settings={
        "kmin":3e-2 * u.Mpc**-1,
        "kmax":2 * u.Mpc**-1,
        "nk":50,
    },
    output=["Power spectrum"]
    )["Power spectrum"]

ssc_CII = scov.SuperSampleCovariance(pobs_CII)
logPlogdb = ssc_CII.logresponse(pobs_CII.k, pobs_CII.z)

In [ ]:
def Lperp(survey: survey_specs.LIMSuvey):
    Sfield = survey.Sfield()
    return np.sqrt(Sfield)

def W2_cube(k, mu, Lperp, Lparr):
    phi = np.linspace(-np.pi, np.pi, 300)
    xperp = (Lperp * k / 2).to(1).value
    xparr = (Lparr * k / 2).to(1).value

    x = xperp[:, None, None] * np.sqrt(np.clip(1 -mu[None, :, None]**2, 0, 1)) * np.cos(phi)
    y = xperp[:, None, None] * np.sqrt(np.clip(1 -mu[None, :, None]**2, 0, 1)) * np.sin(phi)
    z = xparr[:, None] * mu[None, :]

    phi_int = spherical_jn(0, x)**2 * spherical_jn(0, y)**2
    W = spherical_jn(0, z)**2 * np.trapezoid(phi_int, phi, axis=-1) / (2 * np.pi)
    return W


def sigma_survey_intg(Lperp, Lparr, Nt=2000, Nmu=150, alpha=2):
    #Transform logk integral into compactified t
    t = np.linspace(0.0, 1.0, Nt)

    scale = np.minimum(
        Lperp.value,
        Lparr.to(Lperp.unit).value,
        ) * Lperp.unit

    k = ((1 / t - 1) ** alpha) / scale

    mu, w = roots_legendre(Nmu)

    jacobian = alpha / (t * (1.0 - t))

    # mu integration
    T2 = W2_cube(k, mu, Lperp, Lparr)
    T2 = T2.reshape((*k.shape, *mu.shape, *z.shape))
    T2_1d =  np.sum(w[None, :, None] * T2, axis=1) / 2

    # obtain logk integral on t grid
    P = np.reshape(
        mycosmo.matpow(k, z, nonlinear=False, tracer="matter"),
        (*k.shape, *z.shape),
    )
    D = (4 * np.pi * (k[:, None] / (2 * np.pi))**3 * P).to(1).value

    # t integration
    sigma2_intgrnd = D * T2_1d * jacobian[:, None]
    sigma2_intgrnd[~np.logical_and(t>0, t<1), :] = 0.0

    return t, sigma2_intgrnd

def sigma_survey(Lperp, Lparr, Nt=2000, Nmu=150, alpha=2):
    #Transform logk integral into compactified t
    t, sigma2_intgrnd = sigma_survey_intg(Lperp, Lparr, Nt, Nmu, alpha)
    sigma2 = np.trapezoid(sigma2_intgrnd, t, axis=0)
    return sigma2.squeeze()

In [ ]:
sigma2V = sigma_survey(Lperp(pobs_CII.survey_specs), pobs_CII.survey_specs.Lfield())
sigma_over_mu = np.sqrt(sigma2V) * logPlogdb

## Varying $\Omega_{\rm field}$

In [ ]:
Omega_field = np.array([0.0625, 0.25, 1, 4 , 9]) * u.deg**2
L_base = np.sqrt(Omega_field.to(u.sr).value * mycosmo.comoving(z)**2)

In [ ]:
rssc_Omegafield = []
for L in L_base:
    rssc_Omegafield.append(
        np.sqrt(
            sigma_survey(L, pobs_CII.survey_specs.Lfield().squeeze())
        ) * logPlogdb
    )
rssc_Omegafield = np.array(rssc_Omegafield)

In [ ]:
Omega_field_norm = (Omega_field / normalisation["Omega_field"]).to(1).value
rssc_Omegafield_norm = (rssc_Omegafield / sigma_over_mu).mean(-1) # Without varying z there is no k dependence either

In [ ]:
color = iter(Cs)
plt.loglog(Omega_field, rssc_Omegafield_norm, c=next(color), label="SSC prediction")

pobs, pcov = curve_fit(linear, np.log(Omega_field_norm), np.log(rssc_Omegafield_norm))
plt.loglog(Omega_field, np.exp(linear(np.log(Omega_field_norm), *pobs)), c=next(color), label="SSC fit")

plt.loglog(Omega_field, Omega_field_norm**-0.45, c=next(color), label="Concerto fit")

plt.xlabel(r"$\Omega_\mathrm{field}\,[\mathrm{deg}^2]$")
plt.ylabel(r"normalised $\left(\sigma / \mu\right)$")
plt.legend()

In [ ]:
Delta_nu = np.geomspace(5, 20, 5) * u.GHz
rssc_Deltanu = []
for Dnui in Delta_nu:
    specs = surveyspecs_CCATp.copy()
    specs["Delta_nu"] = Dnui

    pobs_CII = myssl.compute(
        myssl.current_cosmology.cosmopars,
        myssl.current_halomodel.haloparams,
        astrodict_CCATp,
        specs,
        pobs_settings={
            "kmin":3e-2 * u.Mpc**-1,
            "kmax":2 * u.Mpc**-1,
            "nk":50,
        },
        output=["Power spectrum"]
        )["Power spectrum"]

    rssc_Deltanu.append(
        np.sqrt(sigma_survey(Lperp(pobs_CII.survey_specs), pobs_CII.survey_specs.Lfield()))
        * logPlogdb
    )
rssc_Deltanu = np.array(rssc_Deltanu)

In [ ]:
Delta_nu_norm = (Delta_nu / normalisation["Delta_nu"]).to(1).value
rssc_Deltanu_norm = (rssc_Deltanu / sigma_over_mu).mean(-1) # Without varying z there is no k dependence either

In [ ]:
pobs, pcov = curve_fit(linear, np.log(Delta_nu_norm), np.log(rssc_Deltanu_norm))

In [ ]:
pobs

In [ ]:
color = iter(Cs)
plt.loglog(Delta_nu, rssc_Deltanu_norm, label="SSC prediction", c=next(color))
plt.loglog(Delta_nu, np.exp(linear(np.log(Delta_nu_norm), *pobs)), label="SSC fit", c=next(color))
plt.loglog(Delta_nu, Delta_nu_norm**-0.62, label="Concerto fit", c=next(color))
plt.legend()
plt.xlabel(r"$\Delta \nu\,[\mathrm{GHz}]$")
plt.ylabel(r"Relative variance $\sigma_V$")

In [ ]:
fw, fh = plt.rcParams['figure.figsize']
fig, axs = plt.subplots(1, 2, figsize=(2 * fw, 1.2 * fh))

color = iter(Cs)
axs[0].loglog(Omega_field, rssc_Omegafield_norm, c=next(color), label="SSC prediction")

pobs, pcov = curve_fit(linear, np.log(Omega_field_norm), np.log(rssc_Omegafield_norm))
axs[0].loglog(Omega_field, np.exp(linear(np.log(Omega_field_norm), *pobs)),
              c=next(color), label="SSC fit")

axs[0].loglog(Omega_field, Omega_field_norm**-0.45,
              c=next(color), label="Concerto fit")

axs[0].set_xlabel(r"$\Omega_\mathrm{field}\,[\mathrm{deg}^2]$")
axs[0].set_ylabel(r"normalised $\left(\sigma / \mu\right)$")

color = iter(Cs)
axs[1].loglog(Delta_nu, rssc_Deltanu_norm,
              c=next(color), label="SSC prediction")

pobs, pcov = curve_fit(linear, np.log(Delta_nu_norm), np.log(rssc_Deltanu_norm))
axs[1].loglog(Delta_nu, np.exp(linear(np.log(Delta_nu_norm), *pobs)),
              c=next(color), label="SSC fit")

axs[1].loglog(Delta_nu, Delta_nu_norm**-0.62,
              c=next(color), label="Concerto fit")

axs[1].set_xlabel(r"$\Delta \nu\,[\mathrm{GHz}]$")

# One shared legend above both subplots
handles, labels = axs[0].get_legend_handles_labels()
fig.legend(handles, labels,
           loc="upper center",
           bbox_to_anchor=(0.5, 1.01),
           ncol=3)

plt.savefig("output/concerto_scaling.pdf")